In [1]:
%pip install qiskit qiskit-aer


In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Create the Bell state |Φ+>
qc.h(0)
qc.cx(0, 1)

# Measure both qubits
qc.measure([0, 1], [0, 1])

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print(counts)
print(qc.draw())

{'11': 491, '00': 533}
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [3]:
qc = QuantumCircuit(2, 2)

qc.h(0)
qc.z(0)
qc.cx(0, 1)

qc.measure([0, 1], [0, 1])

In [4]:
simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print(counts)
print(qc.draw())

{'11': 511, '00': 513}
     ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├┤ Z ├──■──┤M├───
     └───┘└───┘┌─┴─┐└╥┘┌─┐
q_1: ──────────┤ X ├─╫─┤M├
               └───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Prepare |Φ-> = (|00> - |11>)/√2
qc.h(0)
qc.z(0)
qc.cx(0, 1)

# Reveal the phase difference
qc.h(0)
qc.h(1)

qc.measure([0, 1], [0, 1])

result = AerSimulator().run(qc, shots=1024).result()
print(result.get_counts())

{'01': 518, '10': 506}


In [6]:
print(qc.draw())

     ┌───┐┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├┤ Z ├──■──┤ H ├┤M├───
     └───┘└───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ──────────┤ X ├┤ H ├─╫─┤M├
               └───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 


In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def create_bell_state(label):
    qc = QuantumCircuit(2, 2)

    # Create |Phi+>
    qc.h(0)
    qc.cx(0, 1)

    # Select the desired Bell state
    if label in ["Phi-", "Psi-"]:
        qc.z(0)

    if label in ["Psi+", "Psi-"]:
        qc.x(1)

    qc.measure([0, 1], [0, 1])
    return qc

for label in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    qc = create_bell_state(label)
    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=1024).result().get_counts()
    print(label, counts)

Phi+ {'11': 515, '00': 509}
Phi- {'00': 521, '11': 503}
Psi+ {'10': 522, '01': 502}
Psi- {'10': 493, '01': 531}


In [8]:
qc.h(0)
qc.h(1)
qc.measure([0, 1], [0, 1])

In [9]:
print(qc.draw())

     ┌───┐     ┌───┐┌─┐┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤ Z ├┤M├┤ H ├─────┤M├───
     └───┘┌─┴─┐├───┤└╥┘└┬─┬┘┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫──┤M├─┤ H ├─╫─┤M├
          └───┘└───┘ ║  └╥┘ └───┘ ║ └╥┘
c: 2/════════════════╩═══╩════════╩══╩═
                     0   1        0  1 


In [10]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def verify_bell_state(label):
    qc = QuantumCircuit(2, 2)

    # Prepare the Bell state
    qc.h(0)
    qc.cx(0, 1)

    if label in ["Phi-", "Psi-"]:
        qc.z(0)

    if label in ["Psi+", "Psi-"]:
        qc.x(1)

    # Change measurement basis
    qc.h(0)
    qc.h(1)

    # Measure only after all gates
    qc.measure([0, 1], [0, 1])

    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=1024).result().get_counts()

    print(f"{label}: {counts}")
    print(qc.draw())

for label in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    verify_bell_state(label)

Phi+: {'11': 506, '00': 518}
     ┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├──■──┤ H ├┤M├───
     └───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 
Phi-: {'01': 531, '10': 493}
     ┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├┤ H ├┤M├
     └───┘┌─┴─┐├───┤└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ H ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 
Psi+: {'00': 481, '11': 543}
     ┌───┐     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤ H ├─────┤M├───
     └───┘┌─┴─┐├───┤┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 
Psi-: {'10': 501, '01': 523}
     ┌───┐     ┌───┐┌───┐┌─┐   
q_0: ┤ H ├──■──┤ Z ├┤ H ├┤M├───
     └───┘┌─┴─┐├───┤├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 


### Phase Verification of the Four Bell States

I prepared all four Bell states and applied Hadamard gates to both qubits before measurement. This changed the measurement basis and allowed me to distinguish the states by their measurement correlations.

- **Φ⁺:** outcomes `00` and `11`
- **Φ⁻:** outcomes `01` and `10`
- **Ψ⁺:** outcomes `00` and `11`
- **Ψ⁻:** outcomes `01` and `10`

Each outcome pair appeared approximately equally often over 1,024 shots. These results agree with the expected quantum measurement statistics and demonstrate how the measurement basis affects the observable correlations of entangled states.

In [11]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 3)

# Prepare an example input state on Alice's qubit
qc.ry(0.7, 0)

# Create a Bell pair shared by Alice and Bob
qc.h(1)
qc.cx(1, 2)

print(qc.draw())

     ┌─────────┐     
q_0: ┤ Ry(0.7) ├─────
     └──┬───┬──┘     
q_1: ───┤ H ├─────■──
        └───┘   ┌─┴─┐
q_2: ───────────┤ X ├
                └───┘
c: 3/════════════════
                     


In [12]:
qc.cx(0, 1)
qc.h(0)

print(qc.draw())

     ┌─────────┐          ┌───┐
q_0: ┤ Ry(0.7) ├───────■──┤ H ├
     └──┬───┬──┘     ┌─┴─┐└───┘
q_1: ───┤ H ├─────■──┤ X ├─────
        └───┘   ┌─┴─┐└───┘     
q_2: ───────────┤ X ├──────────
                └───┘          
c: 3/══════════════════════════
                               


In [13]:
qc.measure(0, 0)
qc.measure(1, 1)

print(qc.draw())

     ┌─────────┐          ┌───┐┌─┐
q_0: ┤ Ry(0.7) ├───────■──┤ H ├┤M├
     └──┬───┬──┘     ┌─┴─┐└┬─┬┘└╥┘
q_1: ───┤ H ├─────■──┤ X ├─┤M├──╫─
        └───┘   ┌─┴─┐└───┘ └╥┘  ║ 
q_2: ───────────┤ X ├───────╫───╫─
                └───┘       ║   ║ 
c: 3/═══════════════════════╩═══╩═
                            1   0 


In [14]:
import qiskit
print(qiskit.__version__)

2.5.2


In [15]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

teleport_qc = QuantumCircuit(3, 3)

# 1. Prepare Alice's input state
teleport_qc.ry(0.7, 0)

# 2. Create shared Bell pair between qubits 1 and 2
teleport_qc.h(1)
teleport_qc.cx(1, 2)

# 3. Alice's Bell-basis operations
teleport_qc.cx(0, 1)
teleport_qc.h(0)

# 4. Alice measures qubits 0 and 1
teleport_qc.measure(0, 0)
teleport_qc.measure(1, 1)

# 5. Bob applies conditional corrections
with teleport_qc.if_test((teleport_qc.clbits[1], 1)):
    teleport_qc.x(2)

with teleport_qc.if_test((teleport_qc.clbits[0], 1)):
    teleport_qc.z(2)

# 6. Measure Bob's qubit
teleport_qc.measure(2, 2)

print(teleport_qc.draw())


     ┌─────────┐          ┌───┐┌─┐                                          »
q_0: ┤ Ry(0.7) ├───────■──┤ H ├┤M├──────────────────────────────────────────»
     └──┬───┬──┘     ┌─┴─┐└┬─┬┘└╥┘                                          »
q_1: ───┤ H ├─────■──┤ X ├─┤M├──╫───────────────────────────────────────────»
        └───┘   ┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  ┌───┐»
q_2: ───────────┤ X ├───────╫───╫───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─┤ Z ├»
                └───┘       ║   ║   └──╥───  └───┘ ───────┘   └──╥───  └───┘»
                            ║   ║ ┌────╨────┐               ┌────╨────┐     »
c: 3/═══════════════════════╩═══╩═╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞═════»
                            1   0 └─────────┘               └─────────┘     »
«                  
«q_0: ─────────────
«                  
«q_1: ─────────────
«      ───────┐ ┌─┐
«q_2:   End-0 ├─┤M├
«      ───────┘ └╥┘
«c: 3/═══════════╩═
«                2 


In [27]:
simulator = AerSimulator()
compiled = transpile(teleport_qc, simulator)

result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print("Teleportation measurement counts:")
print(counts)

Teleportation measurement counts:
{'100': 22, '101': 28, '110': 35, '111': 39, '011': 222, '000': 230, '010': 220, '001': 228}


In [28]:
bob_zero = sum(
    n for bits, n in counts.items() if bits[0] == "0"
)

bob_one = sum(
    n for bits, n in counts.items() if bits[0] == "1"
)

print("Bob P(0):", bob_zero / 1024)
print("Bob P(1):", bob_one / 1024)

Bob P(0): 0.87890625
Bob P(1): 0.12109375


### Quantum Teleportation: Measurement Results

I implemented a quantum teleportation circuit using Qiskit and AerSimulator. Alice's input state was prepared using an \(R_y(0.7)\) rotation, and a Bell pair was shared between Alice's qubit and Bob's qubit.

Alice performed the Bell-basis operations and measured her two qubits. Bob then applied conditional \(X\) and \(Z\) corrections based on Alice's measurement results before measuring his qubit.

For 1,024 simulation shots, Bob's observed measurement probabilities were approximately:

- **P(0):** 0.879
- **P(1):** 0.121

The theoretical probabilities for the input state are approximately \(P(0)=0.882\) and \(P(1)=0.118\). The small difference is consistent with finite-shot sampling.

This confirms that Bob's measurement statistics agree with the expected statistics of the teleported input state. A separate state-fidelity test would provide a stronger verification of state recovery.


In [17]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# Alice's intended input state
alice_circuit = QuantumCircuit(1)
alice_circuit.ry(0.7, 0)

alice_state = Statevector.from_instruction(alice_circuit)

print("Alice's input state:")
print(alice_state.data)

print("\nExpected measurement probabilities:")
print("P(0) =", abs(alice_state.data[0])**2)
print("P(1) =", abs(alice_state.data[1])**2)

Alice's input state:
[0.93937271+0.j 0.34289781+0.j]

Expected measurement probabilities:
P(0) = 0.8824210936422442
P(1) = 0.11757890635775578


In [31]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport a state with both RY and RZ rotations
teleport_rz_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state
teleport_rz_qc.ry(0.7, 0)
teleport_rz_qc.rz(0.8, 0)

# Create the shared Bell pair
teleport_rz_qc.h(1)
teleport_rz_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_rz_qc.cx(0, 1)
teleport_rz_qc.h(0)

# Alice's measurements
teleport_rz_qc.measure(0, 0)
teleport_rz_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_rz_qc.if_test((teleport_rz_qc.clbits[1], 1)):
    teleport_rz_qc.x(2)

with teleport_rz_qc.if_test((teleport_rz_qc.clbits[0], 1)):
    teleport_rz_qc.z(2)

# Measure Bob
teleport_rz_qc.measure(2, 2)

simulator = AerSimulator()
compiled = transpile(teleport_rz_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation with RY and RZ rotations:")
print(result.get_counts())


Teleportation with RY and RZ rotations:
{'101': 117, '100': 118, '001': 871, '011': 921, '111': 118, '110': 121, '000': 915, '010': 915}


In [32]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport a phase-rotated state and measure in the X basis
teleport_xbasis_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state
teleport_xbasis_qc.ry(0.7, 0)
teleport_xbasis_qc.rz(0.8, 0)

# Create the shared Bell pair
teleport_xbasis_qc.h(1)
teleport_xbasis_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_xbasis_qc.cx(0, 1)
teleport_xbasis_qc.h(0)

# Alice's measurements
teleport_xbasis_qc.measure(0, 0)
teleport_xbasis_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_xbasis_qc.if_test((teleport_xbasis_qc.clbits[1], 1)):
    teleport_xbasis_qc.x(2)

with teleport_xbasis_qc.if_test((teleport_xbasis_qc.clbits[0], 1)):
    teleport_xbasis_qc.z(2)

# Change Bob's measurement basis
teleport_xbasis_qc.h(2)
teleport_xbasis_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_xbasis_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation with X-basis measurement:")
print(result.get_counts())


Teleportation with X-basis measurement:
{'100': 276, '101': 293, '110': 278, '111': 267, '011': 741, '001': 742, '010': 756, '000': 743}


In [33]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport the |+> state
teleport_plus_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state |+>
teleport_plus_qc.h(0)

# Create the shared Bell pair
teleport_plus_qc.h(1)
teleport_plus_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_plus_qc.cx(0, 1)
teleport_plus_qc.h(0)

# Alice's measurements
teleport_plus_qc.measure(0, 0)
teleport_plus_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_plus_qc.if_test((teleport_plus_qc.clbits[1], 1)):
    teleport_plus_qc.x(2)

with teleport_plus_qc.if_test((teleport_plus_qc.clbits[0], 1)):
    teleport_plus_qc.z(2)

# Measure Bob
teleport_plus_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_plus_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation of |+> state:")
print(result.get_counts())


Teleportation of |+> state:
{'110': 563, '101': 526, '011': 509, '111': 529, '001': 522, '100': 484, '010': 463, '000': 500}


In [34]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport |+> and measure Bob in the X basis
teleport_plus_x_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state |+>
teleport_plus_x_qc.h(0)

# Create the shared Bell pair
teleport_plus_x_qc.h(1)
teleport_plus_x_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_plus_x_qc.cx(0, 1)
teleport_plus_x_qc.h(0)

# Alice's measurements
teleport_plus_x_qc.measure(0, 0)
teleport_plus_x_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_plus_x_qc.if_test((teleport_plus_x_qc.clbits[1], 1)):
    teleport_plus_x_qc.x(2)

with teleport_plus_x_qc.if_test((teleport_plus_x_qc.clbits[0], 1)):
    teleport_plus_x_qc.z(2)

# Measure Bob in the X basis
teleport_plus_x_qc.h(2)
teleport_plus_x_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_plus_x_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation of |+> with X-basis measurement:")
print(result.get_counts())


Teleportation of |+> with X-basis measurement:
{'011': 1051, '001': 1060, '000': 983, '010': 1002}
